# DataScienceJobs — Exploration Notebook

Use this notebook to explore the live database, spot trends, and diagnose data quality issues.
All cells connect directly to your Supabase instance via the service key in `.env`.

## Column Provenance

Every column in `job_postings` falls into one of four categories:

| Origin | Meaning |
|---|---|
| 🌐 **API — raw** | Value comes directly from the source API, unchanged |
| 🔧 **API — cleaned** | Comes from the API but is transformed by `data_cleaner.py` before being stored |
| ⚙️ **Pipeline — derived** | Does not exist in the API response; computed from other fields by the pipeline |
| 🗄️ **Database** | Generated by Supabase itself (primary key, timestamps) |

The `v_jobs_enriched` view also adds columns from the `companies` table via a LEFT JOIN (industry, employee_count, etc.) — those are not stored in `job_postings` at all.

In [10]:
import pandas as pd

PROVENANCE = [
    # column                 origin               transformation                                        source file
    ("job_id",               "🌐 API — raw",      "Prefixed by pipeline (adzuna_, serpapi_, etc.)",     "all *_client.py"),
    ("title",                "🌐 API — raw",      "None",                                               "all *_client.py"),
    ("company_name",         "🌐 API — raw",      "None",                                               "all *_client.py"),
    ("company_domain",       "🌐 API — raw",      "Parsed from employer URL; None if unavailable",      "all *_client.py"),
    ("is_remote",            "🌐 API — raw",      "None",                                               "all *_client.py"),
    ("employment_type",      "🌐 API — raw",      "Normalised to FULL_TIME/CONTRACT/etc.",              "all *_client.py"),
    ("salary_currency",      "🌐 API — raw",      "Defaults to CAD",                                    "all *_client.py"),
    ("job_apply_link",       "🌐 API — raw",      "None",                                               "all *_client.py"),
    ("employer_logo",        "🌐 API — raw",      "None",                                               "all *_client.py"),
    ("posted_at",            "🌐 API — raw",      "SerpAPI converts relative dates (UTC); others raw",  "serpapi_client.py"),

    ("job_description",      "🔧 API — cleaned",  "HTML stripped by clean_description()",               "data_cleaner.py"),
    ("location_city",        "🔧 API — cleaned",  "Accent-stripped, alias-resolved, noise removed",     "data_cleaner.py → normalize_city()"),
    ("location_state",       "🔧 API — cleaned",  "Normalised + inferred from city or description",     "data_cleaner.py → normalize_province()"),
    ("location_country",     "🔧 API — cleaned",  "CA → Canada; null → Anywhere",                       "data_cleaner.py → normalize_country()"),
    ("salary_min",           "🔧 API — cleaned",  "Hourly rates converted to annual (×2080)",           "data_cleaner.py → normalize_salary()"),
    ("salary_max",           "🔧 API — cleaned",  "Hourly rates converted to annual (×2080)",           "data_cleaner.py → normalize_salary()"),
    ("salary_period",        "🔧 API — cleaned",  "HOUR → YEAR after conversion",                       "data_cleaner.py → normalize_salary()"),

    ("skills_tags",          "⚙️ Pipeline — derived", "Keyword match against 73-skill list",            "skills_parser.py → extract_skills()"),
    ("years_experience_min", "⚙️ Pipeline — derived", "Regex on description (e.g. '5+ years')",         "data_cleaner.py → extract_years_experience()"),
    ("seniority",            "⚙️ Pipeline — derived", "Title keyword match → years brackets → default Mid", "data_cleaner.py → classify_seniority()"),

    ("id",                   "🗄️ Database",       "UUID primary key, auto-generated",                   "Supabase / PostgreSQL"),
    ("fetched_at",           "🗄️ Database",       "Timestamp of pipeline run (DEFAULT NOW())",          "Supabase / PostgreSQL"),
    ("created_at",           "🗄️ Database",       "Row creation timestamp",                             "Supabase / PostgreSQL"),
    ("updated_at",           "🗄️ Database",       "Auto-updated on every UPDATE via trigger",           "Supabase / PostgreSQL"),
]

prov_df = pd.DataFrame(PROVENANCE, columns=["column", "origin", "transformation", "source file"])

COLOR_MAP = {
    "🌐 API — raw":           "background-color: #1a3a52; color: #90caf9",
    "🔧 API — cleaned":       "background-color: #2d3b1a; color: #a5d6a7",
    "⚙️ Pipeline — derived":  "background-color: #3b2a00; color: #ffcc80",
    "🗄️ Database":            "background-color: #2a1a3b; color: #ce93d8",
}

def _style_row(row):
    style = COLOR_MAP.get(row["origin"], "")
    return [style] * len(row)

prov_df.style.apply(_style_row, axis=1).set_properties(**{"font-size": "12px"}).hide(axis="index")

column,origin,transformation,source file
job_id,🌐 API — raw,"Prefixed by pipeline (adzuna_, serpapi_, etc.)",all *_client.py
title,🌐 API — raw,None,all *_client.py
company_name,🌐 API — raw,None,all *_client.py
company_domain,🌐 API — raw,Parsed from employer URL; None if unavailable,all *_client.py
is_remote,🌐 API — raw,None,all *_client.py
employment_type,🌐 API — raw,Normalised to FULL_TIME/CONTRACT/etc.,all *_client.py
salary_currency,🌐 API — raw,Defaults to CAD,all *_client.py
job_apply_link,🌐 API — raw,None,all *_client.py
employer_logo,🌐 API — raw,None,all *_client.py
posted_at,🌐 API — raw,SerpAPI converts relative dates (UTC); others raw,serpapi_client.py


In [11]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # so pipeline.* imports work from notebooks/

import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

load_dotenv("../.env")

client = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_KEY"])

# ── Load raw job_postings (the source of truth, before the view joins) ────────
resp = client.table("job_postings").select("*").execute()
df = pd.DataFrame(resp.data)

df["posted_at"] = pd.to_datetime(df["posted_at"], utc=True, errors="coerce")
df["fetched_at"] = pd.to_datetime(df["fetched_at"], utc=True, errors="coerce")

# Infer source from job_id prefix
def _source(jid):
    for prefix in ("adzuna_", "theirstack_", "serpapi_"):
        if jid.startswith(prefix):
            return prefix.rstrip("_")
    return "jsearch"

df["source"] = df["job_id"].apply(_source)

print(f"Loaded {len(df):,} rows  ·  {df.columns.size} columns")
df.shape

Loaded 645 rows  ·  25 columns


(645, 25)

In [12]:
# Column types + null counts at a glance
summary = pd.DataFrame({
    "dtype":    df.dtypes,
    "non_null": df.notna().sum(),
    "null":     df.isna().sum(),
    "null_%":   (df.isna().mean() * 100).round(1),
    "sample":   df.iloc[0],
})
summary

,dtype,non_null,null,null_%,sample
id,object,645,0,0.0,9a99bb17-0fe8-4fd7-a40c-9e2c26492426
job_id,object,645,0,0.0,adzuna_5656922219
title,object,645,0,0.0,"Senior Systems Architect (Data / Azure) TOR, ON"
company_name,object,597,48,7.4,Recrute Action
company_domain,object,92,553,85.7,None
location_city,object,408,237,36.7,Church and Wellesley
location_state,object,434,211,32.7,Ontario
location_country,object,645,0,0.0,Canada
is_remote,bool,645,0,0.0,False
employment_type,object,353,292,45.3,FULL_TIME


In [13]:
# Jobs per source
df["source"].value_counts()

source
adzuna        443
jsearch        92
serpapi        62
theirstack     48
Name: count, dtype: int64

In [14]:
# Province distribution
df["location_state"].value_counts(dropna=False)

location_state
Ontario                  261
None                     211
British Columbia          58
Quebec                    46
Alberta                   43
Manitoba                   8
Nova Scotia                6
Saskatchewan               6
New Brunswick              2
Northwest Territories      2
Prince Edward Island       1
Yukon                      1
Name: count, dtype: int64

In [15]:
# Seniority distribution
df["seniority"].value_counts(dropna=False)

seniority
Mid          277
Senior       163
Director+     74
Lead          66
Junior        36
Intern        29
Name: count, dtype: int64

In [17]:
# Top skills across all jobs
from collections import Counter
skill_counts = Counter(sk for tags in df["skills_tags"].dropna() for sk in tags)
pd.Series(skill_counts).sort_values(ascending=False).head(30)

python                 176
machine learning       150
sql                    139
statistics              99
r                       92
azure                   49
forecasting             44
tensorflow              39
pytorch                 39
scikit-learn            38
aws                     37
databricks              37
llm                     37
postgresql              35
generative ai           34
pandas                  33
mysql                   33
spark                   30
power bi                27
feature engineering     25
tableau                 25
gcp                     25
numpy                   25
mongodb                 23
deep learning           21
langchain               20
bigquery                16
time series             16
hadoop                  15
kubernetes              15
dtype: int64

In [18]:
# Browse raw rows — adjust filter as needed
# Examples:
#   df[df["source"] == "serpapi"]
#   df[df["location_state"] == "Quebec"]
#   df[df["salary_min"] > 100_000]
#   df[df["skills_tags"].apply(lambda x: "llm" in (x or []))]

df[df["location_state"] == "Quebec"][["title", "company_name", "location_city", "seniority", "salary_min", "skills_tags"]].head(20)

,title,company_name,location_city,seniority,salary_min,skills_tags
4,Architecte Solutions Données & Analytique / Da...,McKesson,Bas-Saint-Laurent,Lead,85400.0,[]
11,Principal Consultant Principal / Responsable d...,TobogganLabs,Montreal,Lead,NaN,[]
24,Data Scientist,LatentView Analytics,Saint-Hyacinthe,Mid,NaN,"[python, sql, r, machine learning, statistics,..."
45,Data Scientist,LatentView Analytics,Drummondville,Mid,NaN,"[python, sql, r, machine learning, statistics,..."
49,"Senior Data Scientist: AI, ML & NLP Lead",KPI Digital Solutions,Montreal,Senior,NaN,[machine learning]
51,Entry-Level Graduate Analyst - Data Science,WhatJobs Direct,Quebec City,Junior,NaN,"[python, sql, r, pandas, numpy, statistics, ma..."
52,Hybrid Data Scientist - Real-Time AI/ML Systems,Larus Technologies Corporation,Montreal,Mid,NaN,[]
89,"Data Analyst, Quality",Wiraa,Quebec City,Mid,NaN,[]
90,Senior Software Engineer - MAAS,Canonical,Trois-Rivieres,Senior,NaN,[]
91,Senior Software Engineer - MAAS,Canonical,Sherbrooke,Senior,NaN,[]
